In [1]:
import os, time, heapq
import regex as re
from collections import Counter, defaultdict


In [ ]:
## Source from the tiktokenizer 
ROOT = "D:/Tasks/Project_SLM"          # <-- must be the folder containing train/ and heldout/
VOCAB = 48000
SAMPLE_MB = 350                  # per language, byte-BALANCED
MAX_TYPES = None                 # None = no cap (correct; a cap biases Telugu)

In [3]:
# ---- split patterns ---------------------------------------------------------
# cl100k: \p{L}+ excludes combining marks -> shatters Telugu. This is the baseline.
CL100K = r"""'(?i:[sdmt]|ll|ve|re)|[^\r\n\p{L}\p{N}]?+\p{L}++|\p{N}{1,3}+| ?[^\s\p{L}\p{N}]++[\r\n]*+|\s*[\r\n]|\s+(?!\S)|\s"""
 
# o200k: adds \p{M} to the letter classes -> keeps Telugu syllables intact.
O200K = "|".join([
    r"""[^\r\n\p{L}\p{N}]?[\p{Lu}\p{Lt}\p{Lm}\p{Lo}\p{M}]*[\p{Ll}\p{Lm}\p{Lo}\p{M}]+""",
    r"""[^\r\n\p{L}\p{N}]?[\p{Lu}\p{Lt}\p{Lm}\p{Lo}\p{M}]+[\p{Ll}\p{Lm}\p{Lo}\p{M}]*""",
    r"""\p{N}{1,3}""", r""" ?[^\s\p{L}\p{N}]+[\r\n/]*""",
    r"""\s*[\r\n]+""", r"""\s+(?!\S)""", r"""\s+"""])

In [6]:
class Tok:
    def __init__(self, pattern):
        self.pat = re.compile(pattern)
        self.ranks, self.vocab = {}, {i: bytes([i]) for i in range(256)}
        self._cache = {}
 
    def train(self, text, vocab_size=VOCAB, log_every=2000):
        freqs = Counter(self.pat.findall(text))
        print(f"    {len(freqs):,} word types", flush=True)
        if MAX_TYPES and len(freqs) > MAX_TYPES:
            freqs = Counter(dict(freqs.most_common(MAX_TYPES)))
 
        words = [list(w.encode("utf-8")) for w in freqs]
        counts = list(freqs.values())
 
        pairs, where = Counter(), defaultdict(set)
        for wi, w in enumerate(words):
            for p in zip(w, w[1:]):
                pairs[p] += counts[wi]; where[p].add(wi)
 
        # max-heap with lazy deletion; 2nd key reproduces max(key=(count, pair))
        heap = [(-c, (-p[0], -p[1]), p) for p, c in pairs.items()]
        heapq.heapify(heap)
 
        nid, t0 = 256, time.time()
        while nid < vocab_size and heap:
            negc, _, top = heapq.heappop(heap)
            if pairs.get(top, 0) != -negc:          # stale entry
                continue
            if -negc < 2:
                break
            self.ranks[top] = nid
            self.vocab[nid] = self.vocab[top[0]] + self.vocab[top[1]]
            touched = set()
            for wi in list(where[top]):
                w, c = words[wi], counts[wi]
                for p in zip(w, w[1:]):
                    pairs[p] -= c; touched.add(p)
                    if pairs[p] <= 0:
                        pairs.pop(p, None); where.pop(p, None)
                out, i = [], 0
                while i < len(w):
                    if i < len(w) - 1 and (w[i], w[i+1]) == top:
                        out.append(nid); i += 2
                    else:
                        out.append(w[i]); i += 1
                words[wi] = out
                for p in zip(out, out[1:]):
                    pairs[p] += c; where[p].add(wi); touched.add(p)
            for p in touched:
                if p in pairs:
                    heapq.heappush(heap, (-pairs[p], (-p[0], -p[1]), p))
            pairs.pop(top, None); where.pop(top, None)
            nid += 1
            if nid % log_every == 0:
                print(f"    {nid}/{vocab_size}  {time.time()-t0:.0f}s", flush=True)
        return self
 
    def _apply_all(self, ids):
        while len(ids) > 1:
            best = min(zip(ids, ids[1:]), key=lambda p: self.ranks.get(p, 1 << 30))
            if best not in self.ranks:
                break
            n, out, i = self.ranks[best], [], 0
            while i < len(ids):
                if i < len(ids) - 1 and (ids[i], ids[i+1]) == best:
                    out.append(n); i += 2
                else:
                    out.append(ids[i]); i += 1
            ids = out
        return ids
 
    def encode(self, text):
        out = []
        for w in self.pat.findall(text):
            hit = self._cache.get(w)
            if hit is None:
                hit = self._cache[w] = self._apply_all(list(w.encode("utf-8")))
            out.extend(hit)
        return out
 
    def decode(self, ids):
        return b"".join(self.vocab[i] for i in ids).decode("utf-8", errors="replace")
 
    def save(self, prefix):
        with open(prefix + ".model", "w", encoding="utf-8") as f:
            f.write("minbpe v1\n")
            f.write(self.pat.pattern + "\n")
            f.write("0\n")
            for (a, b) in sorted(self.ranks, key=self.ranks.get):
                f.write(f"{a} {b}\n")
 
    def waste(self):
        """fraction of learned tokens that are not valid utf-8"""
        bad = sum(1 for i, v in self.vocab.items()
                  if i >= 256 and "\ufffd" in v.decode("utf-8", errors="replace"))
        return bad / max(len(self.ranks), 1)
 
def balanced_sample(mb=SAMPLE_MB):
    parts = []
    for lang in ("en", "de", "te"):
        cap, got, buf = mb * 1024**2, 0, []
        with open(f"{ROOT}/train/{lang}.txt", encoding="utf-8") as f:
            for line in f:
                buf.append(line); got += len(line.encode())
                if got >= cap:
                    break
        parts.append("".join(buf))
        print(f"  {lang}: {got/1e6:.0f} MB")
    return "\n".join(parts)

def balanced_sample(mb=SAMPLE_MB):
    import random
    rng = random.Random(0)
    parts = []
    for lang in ("en", "de", "te"):
        size = os.path.getsize(f"{ROOT}/train/{lang}.txt")
        cap, got, buf = mb * 1024**2, 0, []
        with open(f"{ROOT}/train/{lang}.txt", encoding="utf-8", errors="replace") as f:
            keep = cap / size                       # sample fraction
            for line in f:
                if rng.random() < keep:
                    buf.append(line); got += len(line.encode())
                    if got >= cap:
                        break
        parts.append("".join(buf))
        print(f"  {lang}: {got/1e6:.0f} MB")
    return "\n".join(parts)

In [7]:
if __name__ == "__main__":
    os.makedirs(f"{ROOT}/tok", exist_ok=True)
    print("sampling...")
    text = balanced_sample()
 
    variants = {"cl100k_pat": CL100K, "o200k_pat": O200K}
    toks = {}
    for name, pat in variants.items():
        print(f"\ntraining {name}", flush=True)
        t = Tok(pat).train(text)
        t.save(f"{ROOT}/tok/{name}")
        toks[name] = t
        print(f"    saved {len(t.ranks):,} merges -> {ROOT}/tok/{name}.model")
 
    probes = {l: open(f"{ROOT}/heldout/{l}.txt", encoding="utf-8").read(2_000_000)
              for l in ("en", "de", "te")}
 
    print("\nchars per token (higher = better compression):")
    print(f"{'variant':<14}{'EN':>7}{'DE':>7}{'TE':>7}{'ratio':>8}{'waste':>8}")
    for name, t in toks.items():
        n = {l: max(len(t.encode(p)), 1) for l, p in probes.items()}
        c = {l: len(probes[l]) / n[l] for l in probes}
        r = max(c.values()) / min(c.values())
        print(f"{name:<14}{c['en']:>7.2f}{c['de']:>7.2f}{c['te']:>7.2f}"
              f"{r:>8.2f}{t.waste():>8.1%}")
 
    print("\nbytes per token (for BPB reference):")
    print(f"{'variant':<14}{'EN':>7}{'DE':>7}{'TE':>7}")
    for name, t in toks.items():
        n = {l: max(len(t.encode(p)), 1) for l, p in probes.items()}
        b = {l: len(probes[l].encode()) / n[l] for l in probes}
        print(f"{name:<14}{b['en']:>7.2f}{b['de']:>7.2f}{b['te']:>7.2f}")
 
    print("\nTelugu token reduction vs cl100k: "
          f"{(1 - len(toks['o200k_pat'].encode(probes['te'])) / len(toks['cl100k_pat'].encode(probes['te']))) * 100:.1f}%")

sampling...
  en: 367 MB
  de: 367 MB
  te: 365 MB

training cl100k_pat
    2,454,619 word types
    2000/48000  515s
    4000/48000  601s
    6000/48000  659s
    8000/48000  699s
    10000/48000  737s
    12000/48000  772s
    14000/48000  808s
    16000/48000  830s
    18000/48000  843s
    20000/48000  865s
    22000/48000  884s
    24000/48000  905s
    26000/48000  922s
    28000/48000  933s
    30000/48000  948s
    32000/48000  958s
    34000/48000  976s
    36000/48000  993s
    38000/48000  1001s
    40000/48000  1007s
    42000/48000  1022s
    44000/48000  1029s
    46000/48000  1035s
    48000/48000  1048s
    saved 47,744 merges -> D:/Tasks/Project_SLM/tok/cl100k_pat.model

training o200k_pat
    3,698,054 word types
    2000/48000  1246s
    4000/48000  1388s
    6000/48000  1472s
    8000/48000  1533s
    10000/48000  1595s
    12000/48000  1636s
    14000/48000  1677s
    16000/48000  1729s
    18000/48000  1752s
    20000/48000  1788s
    22000/48000  1815s
    24000/